### Pydantic 모델
- 정의된 데이터만 전송되도록 요청 바디 검증

In [3]:
from datetime import datetime

from pydantic import BaseModel, PositiveInt,ValidationError


class User(BaseModel):
    id: int  
    name: str = 'John Doe'  
    signup_ts: datetime | None  
    tastes: dict[str, PositiveInt]  


external_data = {
    'id': 123,
    'signup_ts': '2019-06-01 12:22',  
    'tastes': {
        'wine': 9,
        b'cheese': 7,  
        'cabbage': '1',  
    },
}
# user = User(id=1234, signup_ts='2019-06-01 12:22') 의 모양으로 해 줘
user = User(**external_data)  

print(user.id)  
#> 123
print(user.model_dump())  # 모델의 필드와 값을 dict 형태로 변환


123
{'id': 123, 'name': 'John Doe', 'signup_ts': datetime.datetime(2019, 6, 1, 12, 22), 'tastes': {'wine': 9, 'cheese': 7, 'cabbage': 1}}


In [4]:
external_data = {'id': 'not an int', 'tastes': {}}  
try:
    User(**external_data)  
except ValidationError as e:
    print(e.errors())

[{'type': 'int_parsing', 'loc': ('id',), 'msg': 'Input should be a valid integer, unable to parse string as an integer', 'input': 'not an int', 'url': 'https://errors.pydantic.dev/2.13/v/int_parsing'}, {'type': 'missing', 'loc': ('signup_ts',), 'msg': 'Field required', 'input': {'id': 'not an int', 'tastes': {}}, 'url': 'https://errors.pydantic.dev/2.13/v/missing'}]


In [6]:
# Fruit : name - 문자열, color - red or green (문자열)
# weight - 0보다 크고 float, bazam-dict[키-str ,value-[tuple(int,bool,float)]]

from typing import Annotated, Literal
from annotated_types import Gt

#Gt(grater then)
class Fruit(BaseModel):
    name: str
    color:Literal["red","green"]
    weight:Annotated[float,Gt(0)]
    bazam:dict[str,list[tuple[int,bool,float]]]

In [8]:
fruit = Fruit(name='Apple',color='red',weight=4.2,bazam={'foobar':[(1,True,0.1)]})


In [ ]:
fruit = Fruit(name='Apple',color='red',weight=4.2)
fruit

In [11]:
# Book code(4자리), category(소설,과학,역사,경제,자기계발,컴퓨터)
# title-문자, writer-문자, price-int, 0보다 크다

from pydantic import StringConstraints

class Book(BaseModel):
    code:Annotated[str,StringConstraints(min_length=4, max_length=4)]
    category:Literal["소설","과학","역사","경제","자기계발","컴퓨터"]
    title:str
    writer:str
    price:Annotated[int,Gt(0)]

In [13]:
try:
    Book(code='1001',category='경제',price=25000)
except ValidationError as e:
    print(e.errors())

[{'type': 'missing', 'loc': ('title',), 'msg': 'Field required', 'input': {'code': '1001', 'category': '경제', 'price': 25000}, 'url': 'https://errors.pydantic.dev/2.13/v/missing'}, {'type': 'missing', 'loc': ('writer',), 'msg': 'Field required', 'input': {'code': '1001', 'category': '경제', 'price': 25000}, 'url': 'https://errors.pydantic.dev/2.13/v/missing'}]


In [15]:
# User-id:int, name:문자, age 14~100, email, role='user' or 'admin'
# ge, le
from pydantic import Field

class User(BaseModel):
    id:int
    name:str
    age:Annotated[int,Field(ge=14,le=100)]
    email:str
    role:Literal['user','admin']

In [29]:
try:
    User(id=1,name='홍길동',age=14,email='hong@naver.com',role='user')
except ValidationError as e:
    print(e.errors())

In [39]:
# Product : code-문자, name-문자, category-food,clothes,electronics
# price-정수 0보다 커야함, stock-정수 0이상
from annotated_types import Ge, Le

class Product(BaseModel):
    code:str
    name:str
    category:Literal['food','clothes','electronics']
    price:Annotated[int,Gt(0)]
    stock:Annotated[int,Ge(0)]
    #stock:Annotated[int,Field(ge=0)]

In [34]:
try:
    Product(code='1000',name='홍길동',category='food',price=1,stock=0)
except ValidationError as e:
    print(e.errors())

In [46]:
class Item(BaseModel):
    item:str
    status:str

class Cart(BaseModel):
    id:int
    item:Item

In [49]:
try:
    cart = Cart(id=1,item=Item(item='바지',status='판매'))
except ValidationError as e:
    print(e.errors())
    

In [57]:
class OrderItem(BaseModel):
    product_name:str
    quantity:Annotated[int,Gt(0)]
    price:Annotated[int,Gt(0)]

class Order(BaseModel):
    order_id:int
    customer:str
    items:list[OrderItem]

In [58]:
order = Order(order_id=1001,customer='홍길동',items=[
    {"product_name":"노트북","quantity":1,"price":120000},
    {"product_name":"마우스","quantity":1,"price":120000},
])
order

Order(order_id=1001, customer='홍길동', items=[OrderItem(product_name='노트북', quantity=1, price=120000), OrderItem(product_name='마우스', quantity=1, price=120000)])

In [52]:
# Comment : writer-문자, content- 문자
# Post : id-숫자, title-문자,writer-문자, content- 문자,comments:Comment 가 여러개

class Comment(BaseModel):
    writer:str
    content:str

class Post(BaseModel):
    id:int
    title:str
    writer:str
    content:str
    comments:list[Comment]

In [59]:
post = Post(id=1,title="제목",writer="글쓴이",content="내용",comments=[{
    "writer":"글쓴이1","content":"내용1"},
    {"writer":"글쓴이2","content":"내용2"
}])

try:
    post
except ValidationError as e:
    print(e.errors())

post

Post(id=1, title='제목', writer='글쓴이', content='내용', comments=[Comment(writer='글쓴이1', content='내용1'), Comment(writer='글쓴이2', content='내용2')])